### Libraries



In [ ]:
!pip install -q \
  datasets \
  transformers \
  huggingface_hub \
  evaluate

!pip install -U bitsandbytes

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline, DataCollatorWithPadding
from peft import PeftModel
from huggingface_hub import hf_hub_download, login
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score,
from collections import Counter
import json
import time
import psutil
import os

### Load model

In [ ]:
login()

In [ ]:
model_id = "eduhuemar001/distilbert-news/checkpoints/checkpoint-842"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to("cuda")
model.eval()

adapters/epoch_005/adapter_model.safeten(…):   0%|          | 0.00/61.3M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.058, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Lin

### Download test data



In [ ]:
HF_DATASET_REPO = "eduhuemar001/news"

# Download JSON file from HF dataset repo
test_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename="test.json",
    repo_type="dataset",
    local_dir=".",
    local_dir_use_symlinks=False
)

# Load dataset
with open(test_path, "r", encoding="utf-8") as f:
    dataset_test = json.load(f)

### Tokenization

In [ ]:
class NewsPairDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data                  # list of dicts with keys of title, text, status
        self.tok = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        item = self.data[i]
        title = (item.get("title") or "").strip()
        text  = (item.get("text")  or "").strip()
        label = int(item.get("status"))   # 0/1

        enc = self.tok(
            text=title,                   # News title
            text_pair=text,               # News article
            truncation="only_second",     # keep full title, truncate only news article
            max_length=self.max_length,
            padding=False,                # let collator pad
            return_attention_mask=True
        )

        out = {
            "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(label, dtype=torch.long),
        }
        return out

test_tokenized = NewsPairDataset(dataset_test, tokenizer, max_length=512)

### Testing function

In [ ]:
def evaluate_model(model, tokenizer, token_ds):
    instruction = (
        "### Instruction:\n"
        "Klassifiziere die Stimmung der folgenden Bewertung als 'positive', 'neutral' oder 'negative'.\n\n"
        "### Bewertung:\n"
    )
    answer_prefix = "\n\n### Antwort:\n"

    # fixed label order for metrics & confusion matrix
    labels = ["positive", "neutral", "negative"]
    label_set = set(labels)

    print(f"Evaluating TinyLlama model on {device.upper()}")

    pred_labels, true_labels = [], []
    valid_count = 0

    start_inf_wall = time.time()
    if device == "cpu":
        start_inf_cpu = time.process_time()
        process = psutil.Process(os.getpid())
        mem_before = process.memory_info().rss

    # OOM prevention setup
    tokenizer.truncation_side = "left"
    tokenizer.model_max_length = 2048
    model.config.use_cache = False
    model.generation_config.use_cache = False

    for _, row in df.iterrows():
        text = str(row["review_text"])
        true_label = str(row["sentiment"]).lower().strip()

        prompt = instruction + text + answer_prefix
        #inputs = tokenizer(prompt, return_tensors="pt").to(device)
        #with torch.no_grad():
        #    output = model.generate(**inputs, max_new_tokens=2)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                    max_length=2046, padding=False).to(device)
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=2,
                                    do_sample=False, use_cache=False,
                                    pad_token_id=tokenizer.eos_token_id)

        decoded = tokenizer.decode(output[0], skip_special_tokens=True)
        if "### Antwort:" in decoded:
            answer = decoded.split("### Antwort:")[-1].strip().lower()
        else:
            answer = decoded.strip().lower()

        # take first token/word as the label prediction
        pred = (answer.split() + [""])[0]

        # track validity for reporting
        if pred in label_set:
            valid_count += 1
        else:
            # clamp invalid predictions to a neutral fallback for metrics
            pred = "neutral"

        pred_labels.append(pred)
        true_labels.append(true_label)

    end_inf_wall = time.time()
    print(f"Inference wallclock time: {end_inf_wall - start_inf_wall:.2f}s")

    if device == "cpu":
        end_inf_cpu = time.process_time()
        mem_after = process.memory_info().rss
        mem_used_mb = (mem_after - mem_before) / (1024 ** 2)
        print(f"Inference CPU time: {end_inf_cpu - start_inf_cpu:.2f}s")
        print(f"Memory usage increase during inference: {mem_used_mb:.2f} MB")

    total_count = len(pred_labels)
    valid_pct = 100 * valid_count / total_count if total_count else 0.0
    print(f"\nGültige Modellantworten: {valid_count} von {total_count} ({valid_pct:.2f}%)")

    # classification report
    print("\nClassification report:")
    print(classification_report(true_labels, pred_labels, digits=3, zero_division=0))

    # confusion matrix (3x3, ordered by labels)
    cm = confusion_matrix(true_labels, pred_labels, labels=labels)
    cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
    print("\nConfusion matrix:")
    print(cm_df)

    # per-class TP, FP, TN, FN
    tp_fp_tn_fn_rows = []
    N = cm.sum()
    row_sums = cm.sum(axis=1)
    col_sums = cm.sum(axis=0)

    for i, lab in enumerate(labels):
        TP = cm[i, i]
        FP = col_sums[i] - TP
        FN = row_sums[i] - TP
        TN = N - TP - FP - FN
        tp_fp_tn_fn_rows.append(
            {"label": lab, "TP": int(TP), "FP": int(FP), "TN": int(TN), "FN": int(FN)}
        )

    tptn_df = pd.DataFrame(tp_fp_tn_fn_rows).set_index("label")
    print("\n")
    print(tptn_df)
    print("\n")
    return {
        "report": classification_report(true_labels, pred_labels, labels=labels, digits=3, zero_division=0, output_dict=True),
        "confusion_matrix": cm_df,
        "per_class_counts": tptn_df,
        "valid_percentage": valid_pct
    }

### Testing for Epoch 1

In [ ]:
evaluate_model(model, tokenizer, test_tokenized)

Evaluating TinyLlama model on CUDA
Inference wallclock time: 24.10s

Gültige Modellantworten: 83 von 150 (55.33%)

Classification report:
              precision    recall  f1-score   support

    positive      0.325     0.540     0.406        50
     neutral      0.254     0.340     0.291        50
    negative      0.000     0.000     0.000        50

    accuracy                          0.293       150
   macro avg      0.193     0.293     0.232       150
weighted avg      0.193     0.293     0.232       150


Confusion matrix:
               pred_positive  pred_neutral  pred_negative
true_positive             27            23              0
true_neutral              33            17              0
true_negative             23            27              0


          TP  FP   TN  FN
label                    
positive  27  56   44  23
neutral   17  50   50  33
negative   0   0  100  50




{'report': {'positive': {'precision': 0.3253012048192771,
   'recall': 0.54,
   'f1-score': 0.40601503759398494,
   'support': 50.0},
  'neutral': {'precision': 0.2537313432835821,
   'recall': 0.34,
   'f1-score': 0.2905982905982906,
   'support': 50.0},
  'negative': {'precision': 0.0,
   'recall': 0.0,
   'f1-score': 0.0,
   'support': 50.0},
  'accuracy': 0.29333333333333333,
  'macro avg': {'precision': 0.19301084936761972,
   'recall': 0.2933333333333334,
   'f1-score': 0.2322044427307585,
   'support': 150.0},
  'weighted avg': {'precision': 0.19301084936761972,
   'recall': 0.29333333333333333,
   'f1-score': 0.23220444273075855,
   'support': 150.0}},
 'confusion_matrix':                pred_positive  pred_neutral  pred_negative
 true_positive             27            23              0
 true_neutral              33            17              0
 true_negative             23            27              0,
 'per_class_counts':           TP  FP   TN  FN
 label                    
